# Learned against exact equaliser

The trained network in place of the BCJR, $L_0=3$, both families: coded BER on top, the EXIT characteristic below. Both come from the same sweeps, so the only difference is the equaliser. Reads the records under `results/ber` and `results/exit`; BER points resting on fewer than 30 error events are dropped.

In [ ]:
import json, pathlib
import numpy as np
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent
BER = ROOT / "results" / "ber"
EXIT = ROOT / "results" / "exit"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "legend.fontsize": 7, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.3,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": ":",
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

FLOOR = 30       # minimum error events for a BER point to be drawn
TARGET = 1e-4    # where the two equalisers are compared
EB_N0_DB = 4.0   # Eb/N0 of the EXIT panels

def curve(scheme):
    """Eb/N0, BER and the 95% interval for one scheme, ordered and filtered."""
    pts = []
    for f in (BER / scheme).glob("snr_*dB.json"):
        r = json.loads(f.read_text())
        if r["ers_cnt"] >= FLOOR:
            pts.append((r["eb_no_db"], r["ber"], r["ber_lo95"], r["ber_ub95"]))
    pts.sort()
    return [np.array(c) for c in zip(*pts)]

def crossing(eb, ber, target=TARGET):
    """Eb/N0 where the curve crosses `target`, interpolated in log BER."""
    return float(np.interp(-np.log10(target), -np.log10(ber), eb))

def characteristic(name, snr):
    """One EXIT curve from results/exit, matched on value not key spelling."""
    d = json.loads((EXIT / f"{name}.json").read_text())
    key = min(d["results"], key=lambda s: abs(float(s) - snr))
    return np.array(d["IA"]), np.array(d["results"][key]["IE_avg"])

def decoder():
    d = json.loads((EXIT / "coder_K3.json").read_text())
    return np.array(d["IA"]), np.array(d["IE_avg"])

In [ ]:
EQUALISERS = (("", "#000000", "o", "BCJR"), ("_siso", "#d62728", "s", "learned"))

dIA, dIE = decoder()
fig, axes = plt.subplots(2, 2, figsize=(7.6, 7.0))
for col, fam in enumerate(("balanced", "unbalanced")):
    ax = axes[0, col]
    for suffix, color, marker, label in EQUALISERS:
        eb, ber, lo, hi = curve(f"nsm_L3_{fam}_conv_K3_7iters{suffix}")
        ax.errorbar(eb, ber, yerr=[ber - lo, hi - ber], color=color, marker=marker,
                    markersize=3.5, elinewidth=0.7, capsize=1.5,
                    label=f"{label}, {crossing(eb, ber):.2f} dB at $10^{{-4}}$")
    ax.axhline(TARGET, color="0.5", lw=0.7, ls="--")
    ax.set(yscale="log", xlabel="$E_b/N_0$ (dB)", title=f"{fam}, coded, 7 iterations",
           xlim=(0, 7), ylim=(1e-6, 0.5))
    ax.legend(loc="lower left")

    ax = axes[1, col]
    ax.plot(dIE, dIA, color="#9467bd", ls="--", lw=1.2, label="Conv. decoder, $K=3$")
    for suffix, color, _, label in EQUALISERS:
        ia, ie = characteristic(f"nsm_L3_{fam}{suffix}", EB_N0_DB)
        ax.plot(ia, ie, color=color, label=label)
    ax.plot([0, 1], [0, 1], ls=":", lw=0.7, color="0.6")
    ax.set(xlim=(0, 1), ylim=(0, 1), title=f"{fam}, EXIT at {EB_N0_DB:g} dB",
           xlabel=r"$I_A^{\mathrm{modem}} = I_E^{\mathrm{dec}}$")
    ax.legend(loc="lower right")

axes[0, 0].set_ylabel("BER")
axes[1, 0].set_ylabel(r"$I_E^{\mathrm{modem}} = I_A^{\mathrm{dec}}$")
fig.tight_layout()
fig.savefig(FIGURES / "siso_learned.png")

for fam in ("balanced", "unbalanced"):
    at = [crossing(*curve(f"nsm_L3_{fam}_conv_K3_7iters{s}")[:2]) for s, *_ in EQUALISERS]
    print(f"{fam:11s} BCJR {at[0]:.2f} dB   learned {at[1]:.2f} dB   gap {at[1] - at[0]:+.2f} dB at 1e-4")